In [ ]:
!pip install transformers datasets evaluate seqeval --quiet

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

dataset = load_dataset("BramVanroy/conll2003")

label_list = ['O','B-ADJP','I-ADJP','B-ADVP','I-ADVP','B-CONJP','I-CONJP',
              'B-INTJ','I-INTJ','B-LST','I-LST','B-NP','I-NP','B-PP','I-PP',
              'B-PRT','I-PRT','B-SBAR','I-SBAR','B-UCP','I-UCP','B-VP','I-VP']
num_labels = len(label_list)

print("Dataset loaded.")
print("Labels:", label_list)

In [ ]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Tokenization done.")

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label={i: label for i, label in enumerate(label_list)},
    label2id={label: i for i, label in enumerate(label_list)},
    ignore_mismatched_sizes=True
)
print("Model loaded.")

In [ ]:
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./distilbert_chunking_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(1000)),
    eval_dataset=tokenized_dataset["validation"].select(range(200)),
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Training started...")
trainer.train()

In [ ]:
results = trainer.evaluate()

print("\nEvaluation Results")
print("Precision:", round(results["eval_precision"], 4))
print("Recall:   ", round(results["eval_recall"], 4))
print("F1 Score: ", round(results["eval_f1"], 4))
print("Accuracy: ", round(results["eval_accuracy"], 4))

In [ ]:
sentence = "John works at Google in California"
tokens = sentence.split()

inputs = tokenizer(
    tokens,
    is_split_into_words=True,
    return_tensors="pt",
    truncation=True
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

predictions = outputs.logits.argmax(dim=2).squeeze().tolist()
word_ids = tokenizer(tokens, is_split_into_words=True).word_ids()

predicted_labels = []
previous_word_idx = None
for idx, word_idx in enumerate(word_ids):
    if word_idx is None:
        continue
    if word_idx != previous_word_idx:
        predicted_labels.append((tokens[word_idx], label_list[predictions[idx]]))
    previous_word_idx = word_idx

print("\nInference Output")
for token, label in predicted_labels:
    print(f"{token:15} --> {label}")

In [ ]:
print("\nPOS vs Chunking Comparison")
print("• POS Tagging  = grammatical role of each individual word")
print("• Chunking     = phrase grouping like noun phrase / verb phrase")
print("• Chunking is a higher-level task that builds on POS tags")